<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 09 - Geração Automática de Texto </h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2> 1. Introdução</h2>

<p>Enquanto na Extração de Informação a tarefa principal estava relacionada em compreender a escrita e tratar ambiguidades ou similaridades no texto, na geração automática de texto o desafio está em decidir como expressar uma dada informação em palavras.
    </p>
    
  <p>Há diversas aplicações práticas para NLG (do inglês, Natural Language Generation), como:</p>
  
  * Sumarização de documentos;
  * Geração de notícias;
  * Geração de texto para chatbots.
  
  <p>Nesta aula vamos abordar algumas técnicas para geração automática de texto, usando redes neurais recorrentes.</p>

<h2>2. Técnicas para Geração de Texto</h2>

<p> O objetivo da técnica de geração automática de texto é, dada uma sequência de n-gramas (ou caracteres, ou sentenças), prever qual será o próximo n-grama (ou caractere, ou sentença).
    </p>

<h2>3. Redes Neurais Recorrentes</h2>

<p>Uma rede neural recorrente é uma rede com um "loop", ou seja, uma rede que aproveita o aprendizado de uma amostra anterior para amostras seguintes. A figura abaixo, retirada de http://colah.github.io/posts/2015-08-Understanding-LSTMs/ ilustra sua arquitetura.
    </p>
    
![](http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/RNN-unrolled.png)    

<p>Perceba que essa arquitetura de redes neurais é especialmente aplicável para problemas com sequências, uma vez que as amostras anteriores influenciam na aprendizagem.</p>

<p>Porém, as redes recorrentes convencionais tem limitações para utilizar informações aprendidas em etapas mais distantes (que acabam sendo atualizadas durante o treino). As redes neurais LSTM vieram para tentar sanar esse problema, como veremos à seguir.</p>

<h3>Redes LSTM</h3>

<p>As redes LSTM (Long-Short Term Memory, ou de Memória de curto e longo prazo, numa tradução livre), ao contrário das redes neurais recorrentes convencionais, que apenas atualizam o estado aprendido em etapas anteriores, trazem o conceito de memória, que é controlada pelo modelo.
    </p>
    <p>Desta forma, esse modelo não apenas armazena os estados aprendidos em etapas anteriores, mas decide o que deve ser aproveitado, descartado ou atualizado. Vejamos a ilustração abaixo, também extraída de http://colah.github.io/posts/2015-08-Understanding-LSTMs/.
    </p>
    
![](http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-chain.png)    

<p>As etapas ilustradas são: </p>
1. *Qual estado de memória deve ser lembrado*: A primeira camada usa uma função sigmoid que retorna um valor entre 0 e 1 para decidir se o estado de memória vindo da iteração anterior deve ser mantido (1 para completamente mantido) ou esquecido (0 para completamente esquecido);
2. *Qual estado de memória deve ser atualizado*: Primeiramente usa-se uma função sigmoid para decidir quais valores serão atualizados com os novos dados da amostra atual, depois usa-se uma função tanh (que retorna um valor entre -1 e 1) para criar um novo vetor de estados da memória(denominado Ct) que poderá ser utilizado na iteração seguinte;
3. *Atualizar as células de memória antigas*: Combina os estados que devem ser lembrados, os estados que devem ser atualizados e os novos estados para atualizar as células de memória, que serão utilizadas na iteração seguinte;
4. *Decide a saída*: Por fim, a saída da iteração é feita através da combinação de uma função sigmoid e uma função tanh, considerando o estado atual da memória.

    

<h2>4. Usando LSTM para geração de texto</h2>

<p> Vamos nos basear no exemplo disponível em https://www.kaggle.com/shivamb/beginners-guide-to-text-generation-using-lstms, e vamos usar um LSTM para geração de texto.
    </p>

In [ ]:
import keras
from keras.models import Sequential
from keras.layers.recurrent import LSTM
from keras.preprocessing.sequence import pad_sequences
from keras.layers.embeddings import Embedding
from keras.layers.core import Activation, Dense, Dropout
from keras.layers.wrappers import TimeDistributed
from keras.layers.core import Dense, Activation
import keras.utils as kutils

In [ ]:
#import nltk
#nltk.corpus.gutenberg.fileids()


In [ ]:
#raw_txt = nltk.corpus.gutenberg.raw("shakespeare-hamlet.txt") + \
#nltk.corpus.gutenberg.raw("shakespeare-caesar.txt") + \
#nltk.corpus.gutenberg.raw("shakespeare-macbeth.txt")

# Código adaptado de https://www.kaggle.com/shivamb/beginners-guide-to-text-generation-using-lstms
import pandas as pd
import os

curr_dir = '../input/'
all_headlines = []
for filename in os.listdir(curr_dir):
    if 'Articles' in filename:
        article_df = pd.read_csv(curr_dir + filename)
        all_headlines.extend(list(article_df.headline.values))
        break

all_headlines = [h for h in all_headlines if h != "Unknown"]
len(all_headlines)
print(all_headlines[:10])

In [ ]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import string

all_sents = [[w.lower() for w in word_tokenize(sen) if not w in string.punctuation] \
             for sen in all_headlines]

x = []
y = []

print(all_sents[:10])

for sen in all_sents:
    for i in range(1, len(sen)):
        x.append(sen[:i])
        y.append(sen[i])
        

print(x[:10])
print(y[:10])



In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

all_text = [c for sen in x for c in sen]
all_text += [c for c in y]

all_text.append('UNK') # Palavra desconhecida

words = list(set(all_text))
        
word_indexes = {word: index for index, word in enumerate(words)}      

max_features = len(word_indexes)

x = [[word_indexes[c] for c in sen] for sen in x]
y = [word_indexes[c] for c in y]

print(x[:10])
print(y[:10])

y = kutils.to_categorical(y, num_classes=max_features)

maxlen = max([len(sen) for sen in x])

print(maxlen)#tamanho máximo das sentanças


In [ ]:
x = pad_sequences(x, maxlen=maxlen)
x = pad_sequences(x, maxlen=maxlen)

print(x[:10,-10:])
print(y[:10,-10:])

In [ ]:
print(x[:10,-10:])

for y_ in y:
    for i in range(len(y_)):
        if y_[i] != 0:
            print(i)

<p>Agora vamos criar nosso modelo de LSTM usando o Keras.</p>

In [ ]:
embedding_size = 10#elementos no vetor word2vec

model = Sequential()
    
# Add Input Embedding Layer
model.add(Embedding(max_features, embedding_size, input_length=maxlen))
    
# Add Hidden Layer 1 - LSTM Layer
model.add(LSTM(100))#neurônios na camada escondida
model.add(Dropout(0.1))#para não dar overfiting
    
# Add Output Layer
model.add(Dense(max_features, activation='softmax'))#1 neurônio pra cada palavra do dicionario, aqui vai prever a próxima palavra

model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
model.summary()

In [ ]:
model.fit(x, y, epochs=20, verbose=5)#ajusta os pesos 20 vezes

In [ ]:
import pickle

print("Saving model...")
model.save('shak-nlg.h5')

with open('shak-nlg-dict.pkl', 'wb') as handle:
    pickle.dump(word_indexes, handle)

with open('shak-nlg-maxlen.pkl', 'wb') as handle:
    pickle.dump(maxlen, handle)
print("Model Saved!")

In [ ]:
import pickle

model = keras.models.load_model('shak-nlg.h5')
maxlen = pickle.load(open('shak-nlg-maxlen.pkl', 'rb'))
word_indexes = pickle.load(open('shak-nlg-dict.pkl', 'rb'))

<p>Agora vejamos como o algoritmo prevê uma palavra simples.</p>

In [ ]:
sample_seed = input()
sample_seed_vect = np.array([[word_indexes[c] if c in word_indexes.keys() else word_indexes['UNK'] \
                    for c in word_tokenize(sample_seed)]])#se o índice da palavra do input não existir pega do unknow

print(sample_seed_vect)

sample_seed_vect = pad_sequences(sample_seed_vect, maxlen=maxlen)

print(sample_seed_vect)

predicted = model.predict_classes(sample_seed_vect, verbose=0)

print(predicted)

def get_word_by_index(index, word_indexes):
    for w, i in word_indexes.items():
        if index == i:
            return w
        
    return None



for p in predicted:    
    print(get_word_by_index(p, word_indexes))

<p><b>Exercício 9</b>: Escreva um algoritmo que gera palavras até 100 palavras, com base em um texto curto de entrada (no máximo 5 palavras).</p>

In [116]:
def get_word_by_index(index, word_indexes):
    for w, i in word_indexes.items():
        if index == i:
            return w
    return None

In [117]:
sample_seed = input()
sample_seed_vect = [[word_indexes[c] if c in word_indexes.keys() else word_indexes['UNK'] \
                    for c in word_tokenize(sample_seed)]]
    
predict = []
#for i in range(0,101):
while len(sample_seed_vect) < 100:
    
    predicted = model.predict_classes(pad_sequences(sample_seed_vect, maxlen=maxlen,padding='pre'), verbose=0)
    
    #np.concatenate([sample_seed_vect, predicted])
    sample_seed_vect.extend(predicted)
    
res=[]
for w in sample_seed_vect:
    res.append(get_word_by_index(w,word_indexes))

print(' '.join(res))

one


ValueError: `sequences` must be a list of iterables. Found non-iterable: 2404

In [101]:
sample_seed = input()
sample_seed_vect = np.array([[word_indexes[c] if c in word_indexes.keys() else word_indexes['UNK'] \
                    for c in word_tokenize(sample_seed)]])#se o índice da palavra do input não exister pega do unknowow

print(sample_seed_vect)

predict = []

while len(sample_seed_vect) < 100:
    predicted = model.predict_classes(pad_sequences([sample_seed_vect],maxlen=maxlen,padding='pre'),verbose=0)
    #sample_seed_vect.extend(predicted)
    np.concatenate((sample_seed_vect, predicted), axis=0)

res=[]
for w in sample_seed_vect:
    res.append(get_word_by_index(w,word_indexes))
    
print(' '.join(res))

one
[[1209]]


ValueError: Error when checking input: expected embedding_2_input to have 2 dimensions, but got array with shape (1, 21, 1)

In [ ]:
def generate_text(seed_text, next_words, max_sequence_len):
    for j in range(next_words):
        token_list = word_tokenize.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen= 
                             max_sequence_len-1, padding='pre')
        predicted = model.predict_classes(token_list, verbose=0)
  
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [ ]:
print (generate_text("we naughty", 3, 100))